# Notebook 11 - Neural Networks

This notebook adds a small artificial neural network to the SER project. The goal is not to chase the biggest possible number with a deep learning framework. It is to show the mechanics of a dense network on the same tabular audio features used by the classical models: train-only scaling, a hidden ReLU layer, softmax class probabilities, validation loss, and a fair held-out comparison.

The benchmark is the tuned RBF SVM from Notebook 8. That model is a strong classical baseline for this dataset, so the neural-network discussion focuses on training behavior and generalization rather than assuming that more layers automatically win.

In [ ]:
from pathlib import Path
import sys
import time
import warnings

ROOT = Path.cwd()
for candidate in [ROOT, ROOT.parent, ROOT / "serProject", ROOT.parent / "serProject"]:
    if (candidate / "features" / "features.csv").exists():
        ROOT = candidate
        break
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.neural_network import MLPClassifier as SkMLPClassifier
from sklearn.svm import SVC

from minilearn.classifiers import MLPClassifier
from minilearn.metrics import confusion_matrix, plot_confusion_matrix
from minilearn.preprocessing import StandardScaler, train_test_split

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 432
FEATURE_PATH = ROOT / "features" / "features.csv"

## Load data and scale from the training split only

The feature table is already extracted, so the neural network sees the same inputs as the earlier supervised notebooks. Scaling is fit only on the training split. That matters for neural nets because large feature ranges can dominate the gradients, and it matters for evaluation because test-set statistics should stay hidden until scoring.

In [ ]:
META_COLS = {
    "filename",
    "emotion",
    "emotion_id",
    "actor",
    "gender",
    "vocalchannel",
    "intensity",
    "statement",
    "repetition",
    "duration",
}

df = pd.read_csv(FEATURE_PATH)
feature_cols = [c for c in df.columns if c not in META_COLS]
X = df[feature_cols].to_numpy(dtype=float)
y = df["emotion_id"].to_numpy()

emotion_names = (
    df[["emotion_id", "emotion"]]
    .drop_duplicates()
    .sort_values("emotion_id")
    .set_index("emotion_id")["emotion"]
    .to_dict()
)
labels = sorted(emotion_names)
target_names = [emotion_names[i] for i in labels]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"rows={len(df)}, features={len(feature_cols)}, classes={len(labels)}")
print(f"train={len(X_train)}, test={len(X_test)}")
print(emotion_names)

## MiniLearn neural network architecture

The from-scratch network is deliberately small: one hidden layer with ReLU activations, a softmax output layer, cross-entropy loss, mini-batch gradient descent, and L2 regularization. The validation split is taken from the training data inside `fit`, so the held-out test split is still untouched.

In [ ]:
mini_mlp = MLPClassifier(
    hidden_units=48,
    learning_rate=0.01,
    max_iter=160,
    batch_size=64,
    reg_strength=0.001,
    validation_split=0.2,
    random_state=RANDOM_STATE,
)

start = time.perf_counter()
mini_mlp.fit(X_train_s, y_train)
mini_fit_seconds = time.perf_counter() - start

architecture = pd.DataFrame([mini_mlp.architecture_summary()])
architecture["parameters"] = (
    mini_mlp.W1_.size + mini_mlp.b1_.size + mini_mlp.W2_.size + mini_mlp.b2_.size
)
architecture["fit_seconds"] = round(mini_fit_seconds, 4)
architecture

## Training and validation loss

The training curve should trend downward. The validation curve is the more important diagnostic: if training loss keeps falling while validation loss rises, the hidden layer is memorizing quirks of the training split instead of learning emotion patterns that transfer.

In [ ]:
loss_df = pd.DataFrame({
    "epoch": np.arange(1, len(mini_mlp.loss_history_) + 1),
    "training_loss": mini_mlp.loss_history_,
    "validation_loss": mini_mlp.val_loss_history_,
})

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.plot(loss_df["epoch"], loss_df["training_loss"], label="training", color="#246BFE")
ax.plot(loss_df["epoch"], loss_df["validation_loss"], label="validation", color="#D1495B")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("MiniLearn MLP loss curves")
ax.legend()
plt.tight_layout()
plt.show()

loss_df.tail().round(4)

## Neural-network comparison

This cell keeps the comparison modest. It evaluates the from-scratch MiniLearn MLP, a second MiniLearn MLP with stronger regularization, sklearn's optimized MLP implementation, and the validated RBF SVM reference from Notebook 8 using `C=10` and `gamma='scale'`.

In [ ]:
def metric_row(name, model, X_fit, y_fit, X_eval, y_eval, kind="normal"):
    start = time.perf_counter()
    model.fit(X_fit, y_fit)
    fit_seconds = time.perf_counter() - start
    start = time.perf_counter()
    pred = model.predict(X_eval)
    predict_seconds = time.perf_counter() - start
    return {
        "model": name,
        "kind": kind,
        "accuracy": accuracy_score(y_eval, pred),
        "macro_f1": f1_score(y_eval, pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_eval, pred, average="weighted", zero_division=0),
        "fit_seconds": fit_seconds,
        "predict_seconds": predict_seconds,
        "prediction": pred,
        "estimator": model,
    }

model_specs = [
    (
        "MiniLearn MLP 48",
        MLPClassifier(
            hidden_units=48,
            learning_rate=0.01,
            max_iter=160,
            batch_size=64,
            reg_strength=0.001,
            validation_split=0.2,
            random_state=RANDOM_STATE,
        ),
        "neural net",
    ),
    (
        "MiniLearn MLP 64 stronger L2",
        MLPClassifier(
            hidden_units=64,
            learning_rate=0.008,
            max_iter=160,
            batch_size=64,
            reg_strength=0.003,
            validation_split=0.2,
            random_state=RANDOM_STATE,
        ),
        "neural net",
    ),
    (
        "sklearn MLP 64",
        SkMLPClassifier(
            hidden_layer_sizes=(64,),
            activation="relu",
            solver="adam",
            alpha=0.003,
            batch_size=64,
            learning_rate_init=0.001,
            max_iter=300,
            early_stopping=True,
            validation_fraction=0.2,
            n_iter_no_change=20,
            random_state=RANDOM_STATE,
        ),
        "neural net",
    ),
    (
        "validated RBF SVM",
        SVC(kernel="rbf", C=10, gamma="scale", decision_function_shape="ovr", random_state=RANDOM_STATE),
        "classical benchmark",
    ),
]

rows = []
for name, model, kind in model_specs:
    rows.append(metric_row(name, model, X_train_s, y_train, X_test_s, y_test, kind=kind))

results = pd.DataFrame(rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
display_cols = ["model", "kind", "accuracy", "macro_f1", "weighted_f1", "fit_seconds", "predict_seconds"]
results_display = results[display_cols].copy()
for col in ["accuracy", "macro_f1", "weighted_f1", "fit_seconds", "predict_seconds"]:
    results_display[col] = results_display[col].round(4)
results_display

## Neural nets versus the classical benchmark

The RBF SVM is still the reference model because Notebook 8 selected it by cross-validation before the final held-out test check. This plot makes the comparison direct: the neural nets are learning signal, but the kernel method remains a better fit for this small tabular feature table.

In [ ]:
plot_df = results_display.sort_values("macro_f1", ascending=True)
colors = np.where(plot_df["kind"].eq("classical benchmark"), "#2E7D32", "#246BFE")
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.barh(plot_df["model"], plot_df["macro_f1"], color=colors)
ax.set_xlabel("Macro F1")
ax.set_title("Neural networks compared with validated RBF SVM")
ax.set_xlim(0, max(0.9, plot_df["macro_f1"].max() + 0.08))
for i, value in enumerate(plot_df["macro_f1"]):
    ax.text(value + 0.01, i, f"{value:.3f}", va="center")
plt.tight_layout()
plt.show()

## Confusion matrix for the best neural network

The best neural-network row is selected by held-out macro F1 among the neural-net models only. The confusion matrix shows where that network is making tradeoffs across emotions, instead of hiding class-level behavior behind a single average.

In [ ]:
nn_results = results[results["kind"].eq("neural net")].reset_index(drop=True)
best_nn = nn_results.iloc[0]
best_nn_name = best_nn["model"]
best_nn_pred = best_nn["prediction"]

cm = confusion_matrix(y_test, best_nn_pred, labels=labels)
fig, ax = plt.subplots(figsize=(8, 7))
plot_confusion_matrix(cm, target_names, ax=ax, normalize=True, title=f"{best_nn_name} normalized confusion matrix")
plt.tight_layout()
plt.show()

print("Best neural network:", best_nn_name)
print(classification_report(y_test, best_nn_pred, labels=labels, target_names=target_names, zero_division=0))

## Overfitting and generalization notes

- The MiniLearn loss curve gives the expected neural-network pattern: training loss is easier to reduce than validation loss because the optimizer directly sees the training examples.
- L2 regularization and early stopping are useful here because the dataset is small relative to the number of dense weights.
- These inputs are summary audio features in a tabular matrix, not raw waveforms or spectrogram sequences. A dense MLP cannot exploit local time-frequency structure that a CNN or recurrent model might use.
- The RBF SVM remains a strong benchmark because kernel methods often work well on small and medium tabular datasets after scaling.
- A larger neural network could fit the training data more aggressively, but without more data or richer representations that would mainly increase overfitting risk.